In [3]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

#read files
roads   = pd.read_csv("_roads3.csv")
bridges = pd.read_excel("BMMS_overview.xlsx")

# function with Haversine formula to calculate distance between 2 coordinates
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# choose the closest within 25km
def nearest(lat, lon, df, threshold):
    dists = np.array([haversine(lat, lon, r, c) for r, c in zip(df["lat"], df["lon"])])
    i = dists.argmin()
    return (i, dists[i]) if dists[i] <= threshold else (None, None)
# select N1 and N2
n1  = roads[roads["road"] == "N1"].reset_index(drop=True)
n2  = roads[roads["road"] == "N2"].reset_index(drop=True)
#select the close roads
ref = np.vstack([n1[["lat","lon"]].values[::10], n2[["lat","lon"]].values[::10]])
side = roads[~roads["road"].isin(["N1","N2"])].copy()
side = side[side.apply(lambda r: any(haversine(r.lat, r.lon, p[0], p[1]) <= 25 for p in ref), axis=1)]

# preprocess the data of the selected roads
def preprocess(df, road_name):
    df = df[["road","lrp","lat","lon","chainage"]].copy()
    b  = bridges[bridges["road"] == road_name][["LRPName","length","condition","constructionYear"]]
    df = df.merge(b, how="left", left_on="lrp", right_on="LRPName")
    df["model_type"] = np.select([df["length"].notna()], ["bridge"], default="link")
    df.iloc[0,  df.columns.get_loc("model_type")] = "source"
    df.iloc[-1, df.columns.get_loc("model_type")] = "sink"
    df["length"] = df["length"].fillna((df["chainage"].shift(-1) - df["chainage"]) * 1000)
    df["constructionYear"] = pd.to_numeric(df["constructionYear"], errors="coerce")
    df["quality_score"] = df["condition"].map({"A":4,"B":3,"C":2,"D":1}).fillna(0)
    df = df.rename(columns={"lrp":"name"})
    df = df.drop_duplicates("name", keep="last")
    return df.drop(columns=["LRPName","chainage"], errors="ignore").reset_index(drop=True)

n1_p = preprocess(n1, "N1")
n2_p = preprocess(n2, "N2")
side_p = pd.concat([preprocess(g.reset_index(drop=True), r)
                    for r, g in side.groupby("road")], ignore_index=True)


def intersections(df_a, df_b, la, lb):
    seen, rows = set(), []
    for _, rb in df_b.iterrows():
        i, _ = nearest(rb.lat, rb.lon, df_a, 0.5)
        if i is None or i in seen: continue
        seen.add(i)
        ra = df_a.iloc[i]
        rows.append({"road": f"{la}/{lb}", "model_type": "intersection",
                     "name": f"INT_{la}_{lb}_{len(rows)+1}",
                     "lat": (ra.lat+rb.lat)/2, "lon": (ra.lon+rb.lon)/2, "length": 0.0})
    return pd.DataFrame(rows)

ints = pd.concat([
    intersections(n1_p, n2_p,   "N1", "N2"),
    intersections(n1_p, side_p, "N1", "SIDE"),
    intersections(n2_p, side_p, "N2", "SIDE"),
], ignore_index=True)


COLS = ["road","model_type","condition","name","lat","lon","length"]
out  = pd.concat([n1_p, n2_p, side_p, ints], ignore_index=True)
for c in COLS:
    if c not in out.columns: out[c] = np.nan
out = out[COLS].copy()
out.insert(1, "id", out.index + 1)
out[["lat","lon","length"]] = out[["lat","lon","length"]].round(6)

out.to_csv("processed_data.csv", index=False)
print(f"Done: {len(out)} rows\n", out["model_type"].value_counts().to_string())

Done: 14867 rows
 model_type
link            10201
bridge           3920
source            279
sink              279
intersection      188
